# THỰC HÀNH 6: RECURRENT NEURAL NETWORK (BÀI TẬP)

Nội dung thực hiện:
1. Dự báo giá nhà (`raw_sales.csv`)
2. Dự báo giá Bitcoin (`BTC_DATA.csv`)
3. Dự báo điện thế (`household_power_consumption.txt`)
4. Dự báo chứng khoán (`NIFTY_stock_market.csv`)
5. Triển khai Web bằng Flask.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, LSTM, Input
import os

# Hàm tạo dataset theo phương pháp cửa sổ (Sliding Window)
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset)-look_back-1):
        X.append(dataset[i:(i+look_back), 0])
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

def build_and_train(data, target_col, look_back, model_type='RNN', epochs=5):
    values = data[target_col].values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(values)

    X, y = create_dataset(scaled, look_back)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    model = Sequential()
    model.add(Input(shape=(look_back, 1)))
    if model_type == 'LSTM':
        model.add(LSTM(50))
    else:
        model.add(SimpleRNN(50))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=epochs, batch_size=1, verbose=0)
    return model, scaler, X

# --- 1. Dự báo giá nhà ---
if os.path.exists('raw_sales.csv'):
    df_house = pd.read_csv('raw_sales.csv')
else:
    print("Warning: raw_sales.csv not found. Using dummy data.")
    df_house = pd.DataFrame({'price': np.random.randint(400000, 800000, 100)})
model_house, scaler_house, X_house = build_and_train(df_house, 'price', 7, 'RNN')

# --- 2. Dự báo Bitcoin ---
if os.path.exists('BTC_DATA.csv'):
    df_btc = pd.read_csv('BTC_DATA.csv')
else:
    print("Warning: BTC_DATA.csv not found. Using dummy data.")
    df_btc = pd.DataFrame({'priceUSD': np.random.randint(30000, 60000, 100)})
model_btc, scaler_btc, X_btc = build_and_train(df_btc, 'priceUSD', 10, 'LSTM')

# --- 3. Dự báo Điện thế ---
if os.path.exists('household_power_consumption.txt'):
    df_power = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False, na_values=['?']).dropna()
else:
    print("Warning: household_power_consumption.txt not found. Using dummy data.")
    df_power = pd.DataFrame({'Voltage': [230 + np.random.randn() for _ in range(100)]})
model_power, scaler_power, X_power = build_and_train(df_power, 'Voltage', 24, 'RNN')

# --- 4. Dự báo Chứng khoán ---
if os.path.exists('NIFTY_stock_market.csv'):
    df_stock = pd.read_csv('NIFTY_stock_market.csv')
else:
    print("Warning: NIFTY_stock_market.csv not found. Using dummy data.")
    df_stock = pd.DataFrame({'close': np.random.randint(15000, 18000, 100)})
model_stock, scaler_stock, X_stock = build_and_train(df_stock, 'close', 5, 'LSTM')

print("Đã hoàn thành huấn luyện 4 mô hình.")

Đã hoàn thành huấn luyện 4 mô hình.


In [ ]:
# --- 5. Triển khai Flask Web ---
from flask import Flask, render_template_string, jsonify
from google.colab.output import eval_js

app = Flask(__name__)

@app.route('/')
def home():
    return """
    <html><body style='font-family: Arial;'>
    <h2>Dự báo chuỗi thời gian bằng RNN/LSTM</h2>
    <button onclick=\"predict('house')\">Dự báo Giá Nhà</button>
    <button onclick=\"predict('btc')\">Dự báo BTC</button>
    <button onclick=\"predict('power')\">Dự báo Điện thế</button>
    <button onclick=\"predict('stock')\">Dự báo Chứng khoán</button>
    <p id='result'></p>
    <script>
    async function predict(type) {
        const r = await fetch('/predict/'+type);
        const d = await r.json();
        document.getElementById('result').innerText = 'Kết quả dự báo: ' + d.prediction;
    }
    </script>
    </body></html>
    """

@app.route('/predict/<type>')
def api_predict(type):
    if type == 'house': m, data = model_house, X_house
    elif type == 'btc': m, data = model_btc, X_btc
    elif type == 'power': m, data = model_power, X_power
    else: m, data = model_stock, X_stock

    pred = m.predict(data[-1:].reshape(1, data.shape[1], 1), verbose=0)
    return jsonify({'prediction': float(pred[0][0])})

print("Truy cập link: ", eval_js("google.colab.kernel.proxyPort(5000)"))
app.run(port=5000)

Truy cập link:  https://5000-m-s-kkb-usw4b1-m6gxth7wj2h6-b.us-west4-1.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


# THỰC HÀNH 6: RECURRENT NEURAL NETWORK (RNN)

Nội dung bài tập:
1. Dự báo giá nhà (raw_sales.csv)
2. Dự báo giá Bitcoin (BTC_DATA.csv)
3. Dự báo điện thế tiêu thụ (household_power_consumption.txt)
4. Dự báo tỉ giá cổ phiếu (NIFTY_stock_market.csv)
5. Triển khai Web bằng Flask.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input, LSTM
import os

# Hàm hỗ trợ tạo cửa sổ dữ liệu (Sliding Window)
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

# --- 1. DỮ LIỆU GIÁ NHÀ ---
try:
    df_house = pd.read_csv('raw_sales.csv', parse_dates=['datesold']).sort_values('datesold')
except FileNotFoundError:
    df_house = pd.DataFrame({'datesold': pd.date_range('2010-01-01', periods=100), 'price': np.random.randint(400000, 800000, 100)})

# --- 2. DỮ LIỆU BTC ---
try:
    df_btc = pd.read_csv('BTC_DATA.csv', parse_dates=['Date']).sort_values('Date')
except FileNotFoundError:
    df_btc = pd.DataFrame({'Date': pd.date_range('2020-01-01', periods=100), 'priceUSD': np.random.randint(30000, 60000, 100)})

# --- 3. DỮ LIỆU ĐIỆN NĂNG ---
try:
    df_power = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False, na_values=['?']).dropna()
except (FileNotFoundError, pd.errors.ParserError):
    df_power = pd.DataFrame({'Voltage': [230 + np.random.randn() for _ in range(100)]})

# --- 4. DỮ LIỆU CHỨNG KHOÁN ---
try:
    df_stock = pd.read_csv('NIFTY_stock_market.csv', parse_dates=['Date']).sort_values('Date')
except FileNotFoundError:
    df_stock = pd.DataFrame({'Date': pd.date_range('2023-01-01', periods=100), 'close': np.random.randint(15000, 18000, 100)})

print("Đã hoàn thành đọc và chuẩn bị dữ liệu.")

Đã hoàn thành đọc và chuẩn bị dữ liệu.


In [ ]:
# HUẤN LUYỆN CÁC MÔ HÌNH

def train_rnn_model(data, col_name, look_back, rnn_type='RNN'):
    values = data[col_name].values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(values)
    X, y = create_dataset(scaled, look_back)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    model = Sequential()
    model.add(Input(shape=(look_back, 1)))
    if rnn_type == 'LSTM':
        model.add(LSTM(50))
    else:
        model.add(SimpleRNN(50))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=5, verbose=0)
    return model

# Huấn luyện lần lượt
model_house = train_rnn_model(df_house, 'price', 7, 'RNN')
model_btc = train_rnn_model(df_btc, 'priceUSD', 10, 'LSTM')
model_power = train_rnn_model(df_power, 'Voltage', 24, 'RNN')
model_stock = train_rnn_model(df_stock, 'close', 5, 'LSTM')

# Lưu mô hình
model_house.save('house_price_model.h5')
model_btc.save('btc_model.h5')
model_power.save('power_model.h5')
model_stock.save('stock_model.h5')

print("Đã huấn luyện và lưu thành công 4 mô hình.")

Đã huấn luyện và lưu thành công 4 mô hình.


In [ ]:
from flask import Flask, render_template_string, jsonify
from google.colab.output import eval_js
import tensorflow as tf

app = Flask(__name__)

@app.route('/')
def index():
    return """
    <html><body>
    <h2>RNN Forecasting Dashboard</h2>
    <button onclick=\"fetch('/predict/house').then(r=>r.json()).then(d=>alert('House: '+d.res))\">Dự báo Nhà</button>
    <button onclick=\"fetch('/predict/btc').then(r=>r.json()).then(d=>alert('BTC: '+d.res))\">Dự báo BTC</button>
    <button onclick=\"fetch('/predict/power').then(r=>r.json()).then(d=>alert('Power: '+d.res))\">Dự báo Điện</button>
    <button onclick=\"fetch('/predict/stock').then(r=>r.json()).then(d=>alert('Stock: '+d.res))\">Dự báo Chứng khoán</button>
    </body></html>
    """

@app.route('/predict/<m_type>')
def predict(m_type):
    try:
        if m_type == 'house': m = model_house
        elif m_type == 'btc': m = model_btc
        elif m_type == 'power': m = model_power
        else: m = model_stock

        # Giả lập dự báo trên dữ liệu ngẫu nhiên phù hợp shape
        res = m.predict(np.random.rand(1, m.input_shape[1], 1), verbose=0)
        return jsonify({'res': float(res[0][0])})
    except Exception as e: return jsonify({'res': str(e)})

print("Click vào link để xem Web: ", eval_js("google.colab.kernel.proxyPort(5000)"))
app.run(port=5000)

Click vào link để xem Web:  https://5000-m-s-kkb-ase1a1-114mob1u5gd3d-a.asia-east1-1.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


# Lab 5_RNN

## 1. RNN for House Price Prediction (raw_sales.csv)

This section implements a Recurrent Neural Network to forecast house prices based on historical sales data.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input, LSTM

# Hàm hỗ trợ tạo tập dữ liệu dạng cửa sổ (windowing)
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

# --- 1. DỮ LIỆU GIÁ NHÀ ---
try:
    df_house_price = pd.read_csv('raw_sales.csv', parse_dates=['datesold'])
    df_house_price = df_house_price.sort_values('datesold')
except FileNotFoundError:
    dates = pd.date_range(start='2010-01-01', periods=200, freq='D')
    prices = [500000 + (1000 * i) + (np.random.randn() * 20000) for i in range(200)]
    df_house_price = pd.DataFrame({'datesold': dates, 'price': prices})

df_daily_house_price = df_house_price.groupby('datesold')['price'].mean().values.reshape(-1, 1)
scaler_house_price = MinMaxScaler(feature_range=(0, 1))
house_price_scaled = scaler_house_price.fit_transform(df_daily_house_price)
look_back_house_price = 7
X_house_price, y_house_price = create_dataset(house_price_scaled, look_back_house_price)
X_house_price = np.reshape(X_house_price, (X_house_price.shape[0], X_house_price.shape[1], 1))

# --- 2. DỮ LIỆU BTC ---
try:
    df_btc = pd.read_csv('BTC_DATA.csv', parse_dates=['Date'])
except FileNotFoundError:
    dates_btc = pd.date_range(start='2020-01-01', periods=200, freq='D')
    prices_btc = [30000 + (100 * i) + (np.random.randn() * 1000) for i in range(200)]
    df_btc = pd.DataFrame({'Date': dates_btc, 'priceUSD': prices_btc})

btc_values = df_btc['priceUSD'].values.reshape(-1, 1)
scaler_btc = MinMaxScaler(feature_range=(0, 1))
btc_scaled = scaler_btc.fit_transform(btc_values)
look_back_btc = 10
X_btc, y_btc = create_dataset(btc_scaled, look_back_btc)
X_btc = np.reshape(X_btc, (X_btc.shape[0], X_btc.shape[1], 1))

# --- 3. DỮ LIỆU ĐIỆN NĂNG ---
try:
    df_power = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False, na_values=['?'])
    df_power = df_power.dropna()
except FileNotFoundError:
    dates_p = pd.date_range(start='2023-01-01', periods=200, freq='h')
    voltage = [230 + np.random.randn() * 2 for _ in range(200)]
    df_power = pd.DataFrame({'Voltage': voltage})

voltage_values = df_power['Voltage'].values.reshape(-1, 1)
scaler_p = MinMaxScaler(feature_range=(0, 1))
power_scaled = scaler_p.fit_transform(voltage_values)
look_back_p = 24
X_p, y_p = create_dataset(power_scaled, look_back_p)
X_p = np.reshape(X_p, (X_p.shape[0], X_p.shape[1], 1))

# --- 4. DỮ LIỆU CHỨNG KHOÁN ---
try:
    df_stock = pd.read_csv('NIFTY_stock_market.csv', parse_dates=['Date'])
except FileNotFoundError:
    df_stock = pd.DataFrame({'Date': pd.date_range(start='2023-01-01', periods=100), 'close': np.random.randint(15000, 18000, 100)})

stock_values = df_stock['close'].values.reshape(-1, 1)
scaler_s = MinMaxScaler(feature_range=(0, 1))
stock_scaled = scaler_s.fit_transform(stock_values)
look_back_s = 5
X_s, y_s = create_dataset(stock_scaled, look_back_s)
X_s = np.reshape(X_s, (X_s.shape[0], X_s.shape[1], 1))

print("Dữ liệu đã sẵn sàng cho tất cả bài tập.")

Dữ liệu đã sẵn sàng cho tất cả bài tập.


In [15]:
from tensorflow.keras.layers import SimpleRNN, Input, Dense
from tensorflow.keras.models import Sequential

# Xây dựng mô hình RNN cho Giá nhà
model = Sequential([
    Input(shape=(look_back_house_price, 1)),
    SimpleRNN(50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.fit(X_house_price, y_house_price, epochs=5, batch_size=1, verbose=0)
print("House Price Model trained and ready.")

House Price Model trained and ready.


In [14]:
from tensorflow.keras.layers import SimpleRNN, Input, Dense
from tensorflow.keras.models import Sequential

# Xây dựng mô hình RNN cho Giá nhà
model = Sequential([
    Input(shape=(look_back_house_price, 1)),
    SimpleRNN(50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.fit(X_house_price, y_house_price, epochs=5, batch_size=1, verbose=0)
print("House Price Model trained and ready.")

House Price Model trained and ready.


In [13]:
from tensorflow.keras.layers import SimpleRNN, Input, Dense
from tensorflow.keras.models import Sequential

# Xây dựng mô hình RNN cho Giá nhà
model = Sequential([
    Input(shape=(look_back_house_price, 1)),
    SimpleRNN(50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.fit(X_house_price, y_house_price, epochs=5, batch_size=1, verbose=0)
print("House Price Model trained and ready.")

House Price Model trained and ready.


In [ ]:
# Preprocessing: Scaling and Windowing
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

# Aggregate mean price per date
df_daily = df.groupby('datesold')['price'].mean().values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df_daily)

look_back = 7
X, y = create_dataset(data_scaled, look_back)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

print(f"Data prepared. X shape: {X.shape}, y shape: {y.shape}")

Data prepared. X shape: (192, 7, 1), y shape: (192,)


In [ ]:
# Huấn luyện và lưu tất cả mô hình

# 1. House Model
model = Sequential([Input(shape=(look_back, 1)), SimpleRNN(50), Dense(1)])
model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=5, verbose=0)
model.save('house_price_model.h5')

# 2. BTC Model
model_btc = Sequential([Input(shape=(look_back_btc, 1)), LSTM(64), Dense(1)])
model_btc.compile(optimizer='adam', loss='mse')
model_btc.fit(X_btc, y_btc, epochs=5, verbose=0)
model_btc.save('btc_model.h5')

# 3. Power Model
model_power = Sequential([Input(shape=(look_back_p, 1)), SimpleRNN(32), Dense(1)])
model_power.compile(optimizer='adam', loss='mse')
model_power.fit(X_p, y_p, epochs=5, verbose=0)
model_power.save('power_model.h5')

# 4. Stock Model
model_stock = Sequential([Input(shape=(look_back_s, 1)), LSTM(50), Dense(1)])
model_stock.compile(optimizer='adam', loss='mse')
model_stock.fit(X_s, y_s, epochs=5, verbose=0)
model_stock.save('stock_model.h5')

print("Đã huấn luyện và lưu tất cả mô hình thành công.")

Đã huấn luyện và lưu tất cả mô hình thành công.


## 2. RNN for Bitcoin Price Prediction (BTC_DATA.csv)

This section implements an RNN to forecast the USD price of Bitcoin.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Hàm hỗ trợ tạo tập dữ liệu dạng cửa sổ (windowing)
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

try:
    df_btc = pd.read_csv('BTC_DATA.csv', parse_dates=['Date'])
    df_btc = df_btc.sort_values('Date')
    print("Tải thành công tệp BTC_DATA.csv")
except FileNotFoundError:
    print("Không tìm thấy tệp 'BTC_DATA.csv'. Đang tạo dữ liệu BTC giả lập...")
    dates_btc = pd.date_range(start='2020-01-01', periods=200, freq='D')
    prices_btc = [30000 + (100 * i) + (np.random.randn() * 1000) for i in range(200)]
    df_btc = pd.DataFrame({'Date': dates_btc, 'priceUSD': prices_btc})

btc_values = df_btc['priceUSD'].values.reshape(-1, 1)
scaler_btc = MinMaxScaler(feature_range=(0, 1))
btc_scaled = scaler_btc.fit_transform(btc_values)
look_back_btc = 10
X_btc, y_btc = create_dataset(btc_scaled, look_back_btc)
X_btc = np.reshape(X_btc, (X_btc.shape[0], X_btc.shape[1], 1))

display(df_btc.head())

Không tìm thấy tệp 'BTC_DATA.csv'. Đang tạo dữ liệu BTC giả lập...


,Date,priceUSD
0,2020-01-01,31431.391040
1,2020-01-02,29803.020597
2,2020-01-03,31878.038191
3,2020-01-04,31160.267650
4,2020-01-05,28540.999637


In [3]:
from tensorflow.keras.layers import LSTM, Input
from tensorflow.keras.models import Sequential

# Xây dựng mô hình LSTM cho BTC
model_btc = Sequential([
    Input(shape=(look_back_btc, 1)),
    LSTM(64, return_sequences=False),
    Dense(1)
])

model_btc.compile(optimizer='adam', loss='mse')
model_btc.fit(X_btc, y_btc, epochs=10, batch_size=1, verbose=0)
print("BTC Model trained and ready.")

BTC Model trained and ready.


## 3. RNN for Household Power Consumption (household_power_consumption.txt)

This section predicts household voltage using historical power data.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Hàm hỗ trợ tạo tập dữ liệu dạng cửa sổ (windowing)
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

try:
    # Đọc dữ liệu tiêu thụ điện gia đình
    df_power = pd.read_csv('household_power_consumption.txt', sep=';', parse_dates={'datetime': ['Date', 'Time']}, infer_datetime_format=True, low_memory=False, na_values=['?'])
    df_power = df_power.dropna()
    print("Tải thành công tệp household_power_consumption.txt")
except FileNotFoundError:
    print("Không tìm thấy tệp dữ liệu điện. Đang tạo dữ liệu giả lập...")
    dates_p = pd.date_range(start='2023-01-01', periods=200, freq='h')
    voltage = [230 + np.random.randn() * 2 for _ in range(200)]
    df_power = pd.DataFrame({'datetime': dates_p, 'Voltage': voltage})

# Lấy dữ liệu Voltage
voltage_values = df_power['Voltage'].values.reshape(-1, 1)
scaler_p = MinMaxScaler(feature_range=(0, 1))
power_scaled = scaler_p.fit_transform(voltage_values)

look_back_p = 24
X_p, y_p = create_dataset(power_scaled, look_back_p)
X_p = np.reshape(X_p, (X_p.shape[0], X_p.shape[1], 1))

display(df_power.head())

Không tìm thấy tệp dữ liệu điện. Đang tạo dữ liệu giả lập...


/tmp/ipykernel_4768/3091836481.py:16: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df_power = pd.read_csv('household_power_consumption.txt', sep=';', parse_dates={'datetime': ['Date', 'Time']}, infer_datetime_format=True, low_memory=False, na_values=['?'])
/tmp/ipykernel_4768/3091836481.py:16: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_power = pd.read_csv('household_power_consumption.txt', sep=';', parse_dates={'datetime': ['Date', 'Time']}, infer_datetime_format=True, low_memory=False, na_values=['?'])


,datetime,Voltage
0,2023-01-01 00:00:00,231.327032
1,2023-01-01 01:00:00,230.001520
2,2023-01-01 02:00:00,231.517137
3,2023-01-01 03:00:00,229.995444
4,2023-01-01 04:00:00,227.296283


In [5]:
from tensorflow.keras.layers import SimpleRNN, Input, Dense
from tensorflow.keras.models import Sequential

# Xây dựng mô hình RNN cho Power
model_power = Sequential([
    Input(shape=(look_back_p, 1)),
    SimpleRNN(32),
    Dense(1)
])
model_power.compile(optimizer='adam', loss='mse')
model_power.fit(X_p, y_p, epochs=5, batch_size=1, verbose=0)
print("Power Model trained and ready.")

Power Model trained and ready.


## 4. RNN for NIFTY Stock Market (NIFTY_stock_market)

This section predicts the closing price of NIFTY stocks.

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Hàm hỗ trợ tạo tập dữ liệu dạng cửa sổ (windowing)
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

try:
    # Tải dữ liệu chứng khoán NIFTY
    df_stock = pd.read_csv('NIFTY_stock_market.csv', parse_dates=['Date'])
    print("Tải thành công dữ liệu NIFTY")
except FileNotFoundError:
    print("Không tìm thấy dữ liệu chứng khoán. Đang tạo dữ liệu giả lập...")
    df_stock = pd.DataFrame({'Date': pd.date_range(start='2023-01-01', periods=100), 'close': np.random.randint(15000, 18000, 100)})

# Lấy dữ liệu Stock
stock_values = df_stock['close'].values.reshape(-1, 1)
scaler_s = MinMaxScaler(feature_range=(0, 1))
stock_scaled = scaler_s.fit_transform(stock_values)

look_back_s = 5
X_s, y_s = create_dataset(stock_scaled, look_back_s)
X_s = np.reshape(X_s, (X_s.shape[0], X_s.shape[1], 1))

display(df_stock.head())

Không tìm thấy dữ liệu chứng khoán. Đang tạo dữ liệu giả lập...


,Date,close
0,2023-01-01,17862
1,2023-01-02,17859
2,2023-01-03,15456
3,2023-01-04,15736
4,2023-01-05,16771


In [7]:
from tensorflow.keras.layers import LSTM, Input, Dense
from tensorflow.keras.models import Sequential

# Xây dựng mô hình LSTM cho Stock
model_stock = Sequential([
    Input(shape=(look_back_s, 1)),
    LSTM(50, return_sequences=True),
    LSTM(50),
    Dense(1)
])
model_stock.compile(optimizer='adam', loss='mse')
model_stock.fit(X_s, y_s, epochs=5, batch_size=1, verbose=0)
print("Stock Model trained and ready.")

Stock Model trained and ready.


In [16]:
# Lưu tất cả các mô hình đã huấn luyện
model.save('house_price_model.h5')
model_btc.save('btc_model.h5')
model_power.save('power_model.h5')
model_stock.save('stock_model.h5')
print("Tất cả các mô hình đã được lưu thành công.")

Tất cả các mô hình đã được lưu thành công.


## 5. Flask Web Deployment Outline

To deploy these models, create an `app.py` that loads the saved `.h5` models and serves predictions via an API.

In [ ]:
# Example Flask Structure (Pseudo-code)
flask_code = """
from flask import Flask, request, jsonify
import tensorflow as tf

app = Flask(__name__)
model = tf.keras.models.load_model('rnn_model.h5')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json['data']
    prediction = model.predict(data)
    return jsonify({'prediction': prediction.tolist()})

if __name__ == '__main__':
    app.run(debug=True)
"""
print("Example Flask app structure generated.")

Example Flask app structure generated.


### 5. Triển khai giao diện Web với Flask

Đoạn mã dưới đây sẽ tạo ra một ứng dụng Flask hoàn chỉnh. Bạn có thể nhấn chạy, sau đó click vào đường link hiện ra để mở giao diện dự báo trên trình duyệt.

In [ ]:
import os
from flask import Flask, render_template_string, request, jsonify
from google.colab.output import eval_js

# Khởi tạo Flask
app = Flask(__name__)

# Giao diện HTML tích hợp
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>RNN Forecasting Dashboard</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css" rel="stylesheet">
</head>
<body class="container mt-5">
    <h2 class="mb-4">Dự báo RNN - Lab 5</h2>
    <div class="row">
        <!-- Form Câu 1 -->
        <div class="col-md-6 mb-3">
            <div class="card">
                <div class="card-body">
                    <h5 class="card-title">1. Dự báo Giá nhà</h5>
                    <button onclick="predict('house')" class="btn btn-primary">Dự báo ngay</button>
                    <div id="house-res" class="mt-2"></div>
                </div>
            </div>
        </div>
        <!-- Form Câu 2 -->
        <div class="col-md-6 mb-3">
            <div class="card">
                <div class="card-body">
                    <h5 class="card-title">2. Dự báo Bitcoin (USD)</h5>
                    <button onclick="predict('btc')" class="btn btn-warning">Dự báo ngay</button>
                    <div id="btc-res" class="mt-2"></div>
                </div>
            </div>
        </div>
        <!-- Form Câu 3 -->
        <div class="col-md-6 mb-3">
            <div class="card">
                <div class="card-body">
                    <h5 class="card-title">3. Dự báo Điện thế (Voltage)</h5>
                    <button onclick="predict('power')" class="btn btn-success">Dự báo ngay</button>
                    <div id="power-res" class="mt-2"></div>
                </div>
            </div>
        </div>
        <!-- Form Câu 4 -->
        <div class="col-md-6 mb-3">
            <div class="card">
                <div class="card-body">
                    <h5 class="card-title">4. Dự báo Chứng khoán NIFTY</h5>
                    <button onclick="predict('stock')" class="btn btn-info">Dự báo ngay</button>
                    <div id="stock-res" class="mt-2"></div>
                </div>
            </div>
        </div>
    </div>

    <script>
        async function predict(type) {
            document.getElementById(type + '-res').innerText = "Đang tính toán...";
            const response = await fetch('/predict/' + type);
            const data = await response.json();
            document.getElementById(type + '-res').innerText = "Kết quả dự báo: " + data.result;
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(html_template)

@app.route('/predict/<model_type>')
def get_prediction(model_type):
    # Giả lập lấy dữ liệu cuối cùng từ các biến X đã tạo ở các câu trên
    try:
        if model_type == 'house':
            pred = model.predict(X[-1:].reshape(1, 7, 1))[0][0]
        elif model_type == 'btc':
            pred = model_btc.predict(X_btc[-1:].reshape(1, 10, 1))[0][0]
        elif model_type == 'power':
            pred = model_power.predict(X_p[-1:].reshape(1, 24, 1))[0][0]
        elif model_type == 'stock':
            pred = model_stock.predict(X_s[-1:].reshape(1, 5, 1))[0][0]
        return jsonify({'result': round(float(pred), 4)})
    except Exception as e:
        return jsonify({'result': f"Lỗi: {str(e)}"})

# Mở cổng kết nối Colab
print("Đang khởi tạo server... Click vào link bên dưới để mở giao diện:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

if __name__ == '__main__':
    app.run(port=5000)

Đang khởi tạo server... Click vào link bên dưới để mở giao diện:
https://5000-m-s-kkb-usc1a1-1w8ujuc1krt8n-a.us-central1-1.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:22:32] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:22:47] "GET /favicon.ico HTTP/1.1" 404 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step


INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:22:50] "GET /predict/house HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step


INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:23:01] "GET /predict/btc HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step


INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:23:03] "GET /predict/power HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step


INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:23:04] "GET /predict/stock HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2026 12:24:18] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2026 13:07:17] "GET / HTTP/1.1" 200 -
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation